# 02 Feature Exploration

目标：围绕互联网交易欺诈识别的风险特征体系，查看字段族、风险机制、缺失指示变量和候选变量池。


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from fraud_detection.utils import load_yaml
from fraud_detection.data import read_ieee_train, split_by_time
from fraud_detection.features import build_feature_system, candidate_features, add_missing_indicators, categorical_features

config = load_yaml(PROJECT_ROOT / "configs/base.yaml")
df = read_ieee_train(PROJECT_ROOT / config["data"]["raw_dir"])
train, valid, test, split_profile = split_by_time(df)
display(split_profile)


In [ ]:
features = candidate_features(train)
train2, valid2, test2, iv_features, indicators = add_missing_indicators(
    train, valid, test, features, config["features"]["high_missing_threshold"]
)
print("Candidate features:", len(features))
print("IV candidate features:", len(iv_features))
print("Missing indicators:", len(indicators))


In [ ]:
risk_system = build_feature_system(train2)
summary = risk_system.groupby(["FeatureCategory", "RiskMechanism"]).agg(
    FeatureCount=("Feature", "count"),
    AvgMissingRate=("MissingRate", "mean"),
).reset_index().sort_values("FeatureCount", ascending=False)
display(summary)
summary.to_csv(PROJECT_ROOT / "outputs/tables/feature_group_summary.csv", index=False, encoding="utf-8-sig")


In [ ]:
cats = categorical_features(iv_features)
pd.DataFrame({"CategoricalFeature": cats}).to_csv(
    PROJECT_ROOT / "outputs/tables/categorical_feature_candidates.csv", index=False, encoding="utf-8-sig"
)
display(pd.DataFrame({"CategoricalFeature": cats}).head(40))
